# Exp 3.1B: FDA Training-time Augmentation

Train the Siamese U-Net on xBD data augmented with Fourier Domain Adaptation using ida-BD as the style reference.

In [ ]:
# 1. SETUP
import subprocess
subprocess.run(['pip', 'install', '-q', 'segmentation-models-pytorch', 'albumentations', 'shapely'], check=True)
print('Dependencies installed.')


In [ ]:
import os, glob, sys, random, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from shapely import wkt
from shapely.geometry import mapping
import json

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Auto-detect paths
xbd_train_imgs = glob.glob('/kaggle/input/**/train/images', recursive=True)
if not xbd_train_imgs:
    raise Exception('Cannot find xBD train/images')
xbd_root = os.path.dirname(os.path.dirname(xbd_train_imgs[0]))

XBD_TRAIN_IMG   = os.path.join(xbd_root, 'train/images')
XBD_TRAIN_LBL   = os.path.join(xbd_root, 'train/labels')
XBD_VAL_IMG     = os.path.join(xbd_root, 'hold/images')
XBD_VAL_LBL     = os.path.join(xbd_root, 'hold/labels')

ida_imgs = glob.glob('/kaggle/input/**/*post_disaster.png', recursive=True)
ida_imgs = [p for p in ida_imgs if 'xbd' not in p.lower()]
IDA_IMG_DIR = os.path.dirname(ida_imgs[0]) if ida_imgs else '/kaggle/input/ida-bd-images'

ckpts = glob.glob('/kaggle/input/**/*.pth', recursive=True)
CHECKPOINT_PATH = ckpts[0] if ckpts else '/kaggle/input/crossbda-checkpoint/best_model.pth'

OUT_DIR = '/kaggle/working/fda_finetune'
os.makedirs(OUT_DIR, exist_ok=True)

print(f'XBD_TRAIN_IMG: {XBD_TRAIN_IMG}')
print(f'IDA_IMG_DIR:   {IDA_IMG_DIR}')
print(f'CHECKPOINT:    {CHECKPOINT_PATH}')


In [ ]:
# 2. FDA TRANSFORM FUNCTION
def fda_transform(src_img, tgt_img, beta=0.01):
    src_img, tgt_img = src_img.astype(float), tgt_img.astype(float)
    h, w = src_img.shape[:2]
    b = max(1, int(np.floor(min(h, w) * beta)))
    result = np.zeros_like(src_img, dtype=float)
    for c in range(3):
        src_f = np.fft.fft2(src_img[:, :, c])
        tgt_f = np.fft.fft2(tgt_img[:, :, c])
        src_s = np.fft.fftshift(src_f)
        tgt_s = np.fft.fftshift(tgt_f)
        src_amp = np.abs(src_s)
        tgt_amp = np.abs(tgt_s)
        src_pha = np.angle(src_s)
        cy, cx = h // 2, w // 2
        src_amp[cy-b:cy+b, cx-b:cx+b] = tgt_amp[cy-b:cy+b, cx-b:cx+b]
        fused = src_amp * np.exp(1j * src_pha)
        result[:, :, c] = np.real(np.fft.ifft2(np.fft.ifftshift(fused)))
    return np.clip(result, 0, 255).astype(np.uint8)

print('FDA transform function ready.')


In [ ]:
# 3. DATASET WITH FDA AUGMENTATION
def _parse_damage_mask(json_path, img_size=512):
    label_map = {'no-damage': 1, 'minor-damage': 2, 'major-damage': 3, 'destroyed': 4, 'un-classified': 1}
    mask = np.zeros((img_size, img_size), dtype=np.uint8)
    try:
        with open(json_path) as f:
            data = json.load(f)
        for feat in data.get('features', {}).get('xy', []):
            props = feat.get('properties', {})
            cls   = label_map.get(props.get('subtype', ''), 0)
            geom  = wkt.loads(feat['wkt'])
            coords = list(mapping(geom)['coordinates'][0])
            if len(coords) >= 3:
                flat = [(float(x), float(y)) for x, y in coords]
                img_tmp = Image.fromarray(mask)
                ImageDraw.Draw(img_tmp).polygon(flat, fill=int(cls))
                mask = np.array(img_tmp)
    except Exception:
        pass
    return mask

class XBDDatasetFDA(Dataset):
    def __init__(self, img_dir, lbl_dir, ida_img_paths=None, beta=0.01, fda_prob=0.5, img_size=512):
        self.img_dir   = img_dir
        self.lbl_dir   = lbl_dir
        self.ida_paths = ida_img_paths or []
        self.beta      = beta
        self.fda_prob  = fda_prob
        self.img_size  = img_size

        pre_imgs = sorted(glob.glob(os.path.join(img_dir, '*_pre_disaster.png')))
        self.samples = []
        for pre_path in pre_imgs:
            stem      = os.path.basename(pre_path).replace('_pre_disaster.png', '')
            post_path = os.path.join(img_dir, f'{stem}_post_disaster.png')
            lbl_pre   = os.path.join(lbl_dir, f'{stem}_pre_disaster.json')
            lbl_post  = os.path.join(lbl_dir, f'{stem}_post_disaster.json')
            if os.path.exists(post_path) and os.path.exists(lbl_post):
                self.samples.append((pre_path, post_path, lbl_pre, lbl_post))

        self.aug = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ], additional_targets={'image2': 'image'})

        print(f'XBDDatasetFDA: {len(self.samples)} samples | FDA prob={fda_prob} | beta={beta}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pre_p, post_p, lbl_pre_p, lbl_post_p = self.samples[idx]
        pre_img  = np.array(Image.open(pre_p).convert('RGB').resize((self.img_size, self.img_size)))
        post_img = np.array(Image.open(post_p).convert('RGB').resize((self.img_size, self.img_size)))

        if self.ida_paths and random.random() < self.fda_prob:
            tgt_path = random.choice(self.ida_paths)
            tgt_img  = np.array(Image.open(tgt_path).convert('RGB').resize((self.img_size, self.img_size)))
            pre_img  = fda_transform(pre_img, tgt_img, beta=self.beta)
            post_img = fda_transform(post_img, tgt_img, beta=self.beta)

        loc_mask = (_parse_damage_mask(lbl_pre_p, self.img_size) > 0).astype(np.uint8)
        dmg_mask = _parse_damage_mask(lbl_post_p, self.img_size).astype(np.uint8)

        transformed = self.aug(image=pre_img, image2=post_img, mask=dmg_mask)
        pre_t  = transformed['image']
        post_t = transformed['image2']
        mask   = transformed['mask']
        mask_t = mask.long() if isinstance(mask, torch.Tensor) else torch.from_numpy(mask.astype(np.int64)).long()
        loc_t  = torch.from_numpy(loc_mask).long()

        return {'pre_img': pre_t, 'post_img': post_t, 'dmg_mask': mask_t, 'loc_mask': loc_t}

print('XBDDatasetFDA class ready.')


In [ ]:
# 4. MODEL ARCHITECTURE (exact replica of local model.py)
import segmentation_models_pytorch as smp

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)
    def forward(self, x, skip=None):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class SiameseUNet(nn.Module):
    def __init__(self, encoder_name='resnet34', encoder_weights='imagenet', num_damage_classes=5):
        super().__init__()
        self.encoder = smp.encoders.get_encoder(encoder_name, in_channels=3, depth=5, weights=encoder_weights)
        enc_ch = self.encoder.out_channels
        self.bottleneck = DoubleConv(enc_ch[-1] * 2, 512)
        skip_chs = list(reversed([c * 2 for c in enc_ch[1:-1]]))
        dec_out_chs = [256, 128, 64, 32]
        self.decoder = nn.ModuleList()
        in_ch = 512
        for sk, out in zip(skip_chs, dec_out_chs):
            self.decoder.append(DecoderBlock(in_ch, sk, out))
            in_ch = out
        self.final_up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.final_conv = DoubleConv(32, 32)
        self.loc_head   = nn.Conv2d(32, 2, 1)
        self.dmg_head   = nn.Conv2d(32, num_damage_classes, 1)

    def forward_single(self, x):
        return self.encoder(x)

    def forward(self, pre_img, post_img):
        pf = self.forward_single(pre_img)
        qf = self.forward_single(post_img)
        x = self.bottleneck(torch.cat([pf[-1], qf[-1]], dim=1))
        skips = [torch.cat([pf[i], qf[i]], dim=1) for i in range(len(pf) - 2, 0, -1)]
        for block, skip in zip(self.decoder, skips):
            x = block(x, skip)
        x = self.final_conv(self.final_up(x))
        return self.loc_head(x), self.dmg_head(x)

print('SiameseUNet class ready.')


In [ ]:
# 5. LOAD PRETRAINED CHECKPOINT
model = SiameseUNet(encoder_name='resnet34', encoder_weights=None).to(DEVICE)
ckpt  = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
epoch_loaded = ckpt.get('epoch', '?')
score_loaded = ckpt.get('score', 0)
print(f'Loaded from epoch {epoch_loaded} (score: {score_loaded:.4f})')


In [ ]:
# 6. TRAINING CONFIG AND DATALOADERS
FDA_BETA   = 0.01
FDA_PROB   = 0.5
NUM_EPOCHS = 20
LR         = 2e-4
BATCH_SIZE = 8
IMG_SIZE   = 512

ida_post_imgs = sorted(glob.glob(os.path.join(IDA_IMG_DIR, '*_post_disaster.png')))
print(f'ida-BD reference images: {len(ida_post_imgs)}')

train_ds = XBDDatasetFDA(XBD_TRAIN_IMG, XBD_TRAIN_LBL, ida_post_imgs, FDA_BETA, FDA_PROB, IMG_SIZE)
val_ds   = XBDDatasetFDA(XBD_VAL_IMG,   XBD_VAL_LBL,   [],            FDA_BETA, 0.0,      IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')


In [ ]:
# 7. LOSS AND OPTIMIZER
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

ce_loss   = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

def compute_f1(preds, targets, num_classes):
    f1s = []
    for c in range(1, num_classes):
        tp = ((preds == c) & (targets == c)).sum().float()
        fp = ((preds == c) & (targets != c)).sum().float()
        fn = ((preds != c) & (targets == c)).sum().float()
        f1s.append((2 * tp / (2 * tp + fp + fn + 1e-8)).item())
    return float(np.mean(f1s)) if f1s else 0.0

print('Loss and optimizer ready.')


In [ ]:
# 8. TRAINING LOOP
best_score = 0.0
save_path  = os.path.join(OUT_DIR, 'fda_best_model.pth')

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Train]', leave=False):
        pre    = batch['pre_img'].to(DEVICE, non_blocking=True)
        post   = batch['post_img'].to(DEVICE, non_blocking=True)
        loc_gt = batch['loc_mask'].to(DEVICE, non_blocking=True)
        dmg_gt = batch['dmg_mask'].to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        loc_pred, dmg_pred = model(pre, post)
        loss = 0.4 * ce_loss(loc_pred, loc_gt) + 0.6 * ce_loss(dmg_pred, dmg_gt)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()
    train_loss /= len(train_loader)

    model.eval()
    lp_list, lg_list, dp_list, dg_list = [], [], [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Val]', leave=False):
            pre  = batch['pre_img'].to(DEVICE)
            post = batch['post_img'].to(DEVICE)
            loc_pred, dmg_pred = model(pre, post)
            lp_list.append(loc_pred.argmax(1).cpu())
            lg_list.append(batch['loc_mask'])
            dp_list.append(dmg_pred.argmax(1).cpu())
            dg_list.append(batch['dmg_mask'])

    f1_loc = compute_f1(torch.cat(lp_list), torch.cat(lg_list), 2)
    f1_dmg = compute_f1(torch.cat(dp_list), torch.cat(dg_list), 5)
    score  = 0.3 * f1_loc + 0.7 * f1_dmg

    print(f'Epoch {epoch:02d} | Loss: {train_loss:.4f} | F1_loc: {f1_loc:.4f} | F1_dmg: {f1_dmg:.4f} | xView2: {score:.4f}')

    if score > best_score:
        best_score = score
        torch.save({'model_state': model.state_dict(), 'epoch': epoch, 'score': score}, save_path)
        print(f'  - Saved best model (score={score:.4f})')

print(f'Training complete. Best xView2 score: {best_score:.4f}')
print(f'Checkpoint saved to: {save_path}')
